<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/03_construction_eda_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DADS5001 Mini Project
## การสำรวจโครงการจ้างก่อสร้างและการกำหนดกลุ่มศึกษา ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

Notebook นี้เริ่มจากโครงการจ้างก่อสร้างทั้งหมด เพื่อสำรวจขนาดโครงการและวิธีจัดซื้อก่อนนำกฎหมายมาอธิบายเส้นวงเงิน 500,000 บาท จากนั้นจึงกำหนดกลุ่มศึกษาหลักเป็น **โครงการวิธีเฉพาะเจาะจงที่มีวงเงินไม่เกิน 500,000 บาท**

### คำถาม

1. โครงการส่วนใหญ่มีขนาดเท่าใด และจำนวนโครงการกับวงเงินรวมให้ภาพเหมือนกันหรือไม่
2. จุดกระจุกของวงเงินสัมพันธ์กับวิธีจัดซื้ออย่างไร
3. กฎหมายอธิบายความสำคัญของเส้น 500,000 บาทอย่างไร
4. จากโครงการก่อสร้างทั้งหมด กลุ่มศึกษาหลักเหลือกี่โครงการ
5. ราคาที่ตกลงสัมพันธ์กับงบประมาณและราคากลางอย่างไร

> ผล EDA ใช้สร้างคำถามและกำหนดขอบเขต ไม่ใช่หลักฐานว่ามีการกระทำผิด


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ดาวน์โหลดฟอนต์ TH Sarabun New
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# เพิ่มฟอนต์ให้ Matplotlib
fm.fontManager.addfont(
    'thsarabunnew-webfont.ttf'
)

# กำหนดฟอนต์เริ่มต้นสำหรับ Matplotlib และ Seaborn
mpl.rc(
    'font',
    family='TH Sarabun New'
)
mpl.rcParams['axes.unicode_minus'] = False

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option(
    'display.float_format',
    lambda value: f'{value:,.2f}'
)

data_path = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract/'
    'processed/construction_contracts_2569.csv'
)

project_directory = data_path.parents[4]
figure_directory = project_directory / 'figure'
figure_directory.mkdir(parents=True, exist_ok=True)

construction_data = pd.read_csv(
    data_path,
    low_memory=False
)

print(f'Shape: {construction_data.shape}')
print(f'Figure directory: {figure_directory}')


## 1. ทำความเข้าใจข้อมูล

เริ่มจากตรวจสอบโครงสร้าง ชนิดข้อมูล ค่าว่าง และจำนวนค่าที่ไม่ซ้ำ
ของแต่ละคอลัมน์ เพื่อเลือกตัวแปรที่สามารถนำมาใช้วิเคราะห์ได้จริง

การตรวจในส่วนนี้มุ่งเฉพาะข้อมูลที่จำเป็นต่อ EDA
ไม่ใช่การตรวจสอบคุณภาพข้อมูลอย่างละเอียด

In [ ]:
display(construction_data.head())

In [ ]:
field_summary = pd.DataFrame({
    'column': construction_data.columns,
    'dtype': (
        construction_data
        .dtypes
        .astype(str)
        .values
    ),
    'non_null': (
        construction_data
        .notna()
        .sum()
        .values
    ),
    'missing': (
        construction_data
        .isna()
        .sum()
        .values
    ),
    'missing_pct': (
        construction_data
        .isna()
        .mean()
        .mul(100)
        .values
    ),
    'unique': (
        construction_data
        .nunique(
            dropna=True
        )
        .values
    )
})

display(field_summary)

### สรุปการทำความเข้าใจข้อมูล

ข้อมูลจ้างก่อสร้างมี 180,079 แถว และ 29 คอลัมน์
ตัวแปรหลักด้านงบประมาณ ราคา วิธีจัดซื้อ หน่วยงาน และพื้นที่
มีข้อมูลค่อนข้างครบถ้วน

คอลัมน์ `ราคากลาง (บาท)` ขาดข้อมูล 107 แถว หรือ 0.06%
และ `วงเงินงบประมาณในสัญญา (บาท)` ขาดข้อมูล 90 แถว
หรือ 0.05% ส่วนข้อมูลพิกัดขาด 1,438 แถว หรือ 0.80%

คอลัมน์ต่อไปนี้มีเพียงค่าเดียว จึงไม่ช่วยในการเปรียบเทียบ
ภายในชุดข้อมูลจ้างก่อสร้าง:

- ชื่อประเภทโครงการ
- ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง
- ปีงบประมาณ
- สถานะโครงการ
- สถานะสัญญา

ก่อนสร้างข้อมูลระดับโครงการ จะตรวจสอบว่าแถวที่ใช้
`รหัสโครงการ` เดียวกันมีค่าของตัวแปรระดับโครงการ
สอดคล้องกันหรือไม่

In [ ]:
project_columns = [
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อหน่วยงาน',
    'ชื่อหน่วยงานย่อย',
    'ชื่อวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    ),
    'วันที่เกิดรายการ',
    'จังหวัด',
    'เขต/อำเภอ',
    'แขวง/ตำบล'
]

project_consistency = (
    construction_data
    .groupby('รหัสโครงการ')[
        project_columns
    ]
    .nunique(
        dropna=False
    )
)

inconsistent_projects = (
    project_consistency
    .gt(1)
    .sum()
    .rename(
        'projects_with_multiple_values'
    )
    .to_frame()
)

display(inconsistent_projects)

## 2. เตรียมข้อมูลระดับโครงการ

จากการตรวจสอบโครงการที่ปรากฏมากกว่าหนึ่งแถว พบว่าตัวแปร
ระดับโครงการทั้งหมดมีค่าตรงกันภายใน `รหัสโครงการ` เดียวกัน
ได้แก่ ชื่อโครงการ หน่วยงาน วิธีจัดซื้อ วันที่ ราคา และพื้นที่

ดังนั้น สามารถสร้างชุดข้อมูลระดับโครงการโดยเก็บหนึ่งแถวต่อ
`รหัสโครงการ` ได้ โดยไม่ทำให้ข้อมูลระดับโครงการสูญหาย

ชุดข้อมูลที่ใช้ต่อจากนี้แบ่งเป็น:

- `project_data` สำหรับวิเคราะห์จำนวนโครงการ ราคา หน่วยงาน
  วิธีจัดซื้อ และพื้นที่
- `construction_data` สำหรับวิเคราะห์ผู้ชนะและรายละเอียดสัญญา

In [ ]:
project_data = (
    construction_data
    .drop_duplicates(
        subset='รหัสโครงการ',
        keep='first'
    )
    .copy()
)

print(
    f'Contract-level rows: '
    f'{len(construction_data):,}'
)

print(
    f'Project-level rows: '
    f'{len(project_data):,}'
)

print(
    f'Rows removed from project-level analysis: '
    f'{len(construction_data) - len(project_data):,}'
)

In [ ]:
money_columns = [
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    )
]

money_summary = (
    project_data[
        money_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

display(money_summary)

non_positive_summary = pd.DataFrame({
    'column': money_columns,
    'zero_or_negative': [
        project_data[column]
        .le(0)
        .sum()
        for column in money_columns
    ],
    'missing': [
        project_data[column]
        .isna()
        .sum()
        for column in money_columns
    ]
})

display(non_positive_summary)

### สิ่งที่พบจากตัวแปรด้านการเงิน

วงเงินงบประมาณมีค่ามัธยฐาน 395,000 บาท แต่มีค่าเฉลี่ยประมาณ 2.66 ล้านบาท เพราะโครงการมูลค่าสูงจำนวนน้อยดึงค่าเฉลี่ยขึ้น

ค่ามัธยฐานจึงอธิบายขนาดของโครงการทั่วไปได้ดีกว่าค่าเฉลี่ย ส่วนถัดไปต้องอ่านสัดส่วนจำนวนโครงการคู่กับสัดส่วนวงเงินรวม เพื่อไม่ให้โครงการขนาดเล็กจำนวนมากหรือโครงการใหญ่มูลค่าสูงครอบงำการตีความเพียงด้านเดียว


In [ ]:
budget_column = 'วงเงินงบประมาณ (บาท)'
reference_price_column = 'ราคากลาง (บาท)'

awarded_price_column = (
    'ราคาที่ตกลงซื้อ / จ้าง '
    'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
)

# Positive value means the awarded price is below the budget
project_data['budget_saving'] = (
    project_data[budget_column]
    - project_data[awarded_price_column]
)

project_data['budget_saving_pct'] = (
    project_data['budget_saving']
    .div(project_data[budget_column])
    .mul(100)
)

# Positive value means the awarded price is below the reference price
project_data['reference_discount'] = (
    project_data[reference_price_column]
    - project_data[awarded_price_column]
)

project_data['reference_discount_pct'] = (
    project_data['reference_discount']
    .div(
        project_data[
            reference_price_column
        ]
    )
    .mul(100)
)

In [ ]:
price_difference_columns = [
    'budget_saving',
    'budget_saving_pct',
    'reference_discount',
    'reference_discount_pct'
]

display(
    project_data[
        price_difference_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

price_relationship_summary = pd.DataFrame({
    'comparison': [
        'Awarded price below budget',
        'Awarded price equal to budget',
        'Awarded price above budget',
        'Awarded price below reference price',
        'Awarded price equal to reference price',
        'Awarded price above reference price'
    ],
    'project_count': [
        project_data['budget_saving'].gt(0).sum(),
        project_data['budget_saving'].eq(0).sum(),
        project_data['budget_saving'].lt(0).sum(),
        project_data['reference_discount'].gt(0).sum(),
        project_data['reference_discount'].eq(0).sum(),
        project_data['reference_discount'].lt(0).sum()
    ]
})

price_relationship_summary['project_pct'] = (
    price_relationship_summary['project_count']
    .div(len(project_data))
    .mul(100)
)

display(price_relationship_summary)

### การตรวจความสัมพันธ์ของราคาเบื้องต้น

ในโครงการก่อสร้างทั้งหมด ราคาที่ตกลงต่ำกว่างบประมาณ 67.42% เท่ากับงบประมาณ 32.41% และสูงกว่า 0.18% ส่วนการเทียบราคากลางใช้เป็นการตรวจเบื้องต้น เพราะยังมีราคากลางที่ขาดหายหรือผิดสัดส่วน

ผลส่วนนี้ใช้ตรวจโครงสร้างข้อมูลเท่านั้น การสร้างตัวชี้วัดราคาจะจำกัดให้เหลือกลุ่มศึกษาหลักและราคากลางที่ใช้งานได้


## 3. การวิเคราะห์เชิงสำรวจ

### 3.1 โครงการส่วนใหญ่อยู่ช่วงใด และเงินส่วนใหญ่อยู่ช่วงใด

เริ่มจากโครงการจ้างก่อสร้างทั้งหมด เปรียบเทียบสัดส่วนจำนวนโครงการกับสัดส่วนวงเงินรวมในแต่ละช่วง เพื่อค้นหาจุดที่ควรอธิบายต่อด้วยวิธีจัดซื้อและกฎหมาย


In [ ]:
project_data['budget_million'] = (
    project_data[budget_column]
    .div(1_000_000)
)

project_data['log10_budget'] = np.log10(
    project_data[budget_column]
)

budget_p99 = (
    project_data['budget_million']
    .quantile(0.99)
)

print(
    f'99th percentile of budget: '
    f'{budget_p99:,.2f} million THB'
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.histplot(
    data=project_data,
    x='budget_million',
    bins=50,
    ax=axes[0],
    color='#4C78A8'
)

axes[0].set_xlim(0, budget_p99)
axes[0].set_title('การกระจายวงเงินไม่เกินเปอร์เซ็นไทล์ที่ 99')
axes[0].set_xlabel('วงเงินงบประมาณ (ล้านบาท)')
axes[0].set_ylabel('จำนวนโครงการ')

sns.histplot(
    data=project_data,
    x='log10_budget',
    bins=50,
    ax=axes[1],
    color='#E67E22'
)

axes[1].set_title('การกระจายวงเงินบนมาตราส่วนลอการิทึม')
axes[1].set_xlabel('Log10 ของวงเงินงบประมาณ (บาท)')
axes[1].set_ylabel('จำนวนโครงการ')

plt.tight_layout()
plt.show()


In [ ]:
common_budget_values = (
    project_data[budget_column]
    .value_counts()
    .head(15)
    .rename_axis('budget')
    .reset_index(name='project_count')
)

common_budget_values['budget_million'] = (
    common_budget_values['budget']
    .div(1_000_000)
)

common_budget_values['project_pct'] = (
    common_budget_values['project_count']
    .div(len(project_data))
    .mul(100)
)

display(
    common_budget_values[
        [
            'budget',
            'budget_million',
            'project_count',
            'project_pct'
        ]
    ]
)

### ข้อค้นพบจากการกระจายของวงเงิน

วงเงินกระจายเบ้ขวาและเปอร์เซ็นไทล์ที่ 99 อยู่ที่ 32.50 ล้านบาท ภาพมาตราส่วนลอการิทึมใช้ตรวจรูปทรงการกระจาย ส่วนภาพช่วงงบประมาณด้านล่างใช้สื่อสารข้อค้นพบหลัก


In [ ]:
budget_bins = [
    0,
    100_000,
    200_000,
    300_000,
    400_000,
    500_000,
    1_000_000,
    5_000_000,
    10_000_000,
    50_000_000,
    np.inf
]

budget_labels = [
    'ไม่เกิน 100,000',
    '100,001–200,000',
    '200,001–300,000',
    '300,001–400,000',
    '400,001–500,000',
    '500,001–1 ล้าน',
    'มากกว่า 1–5 ล้าน',
    'มากกว่า 5–10 ล้าน',
    'มากกว่า 10–50 ล้าน',
    'มากกว่า 50 ล้าน'
]

project_data['budget_band'] = pd.cut(
    project_data[budget_column],
    bins=budget_bins,
    labels=budget_labels,
    include_lowest=True,
    right=True
)

budget_band_summary = (
    project_data
    .groupby('budget_band', observed=False)
    .agg(
        project_count=('รหัสโครงการ', 'size'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

budget_band_summary['project_pct'] = (
    budget_band_summary['project_count']
    / len(project_data)
    * 100
)

budget_band_summary['budget_pct'] = (
    budget_band_summary['total_budget']
    / budget_band_summary['total_budget'].sum()
    * 100
)

display(budget_band_summary)


In [ ]:
budget_band_plot = (
    budget_band_summary[
        ['budget_band', 'project_pct', 'budget_pct']
    ]
    .melt(
        id_vars='budget_band',
        var_name='measure',
        value_name='percentage'
    )
)

budget_band_plot['measure'] = budget_band_plot['measure'].map({
    'project_pct': 'สัดส่วนจำนวนโครงการ',
    'budget_pct': 'สัดส่วนวงเงินรวม'
})

fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    data=budget_band_plot,
    y='budget_band',
    x='percentage',
    hue='measure',
    palette=['#4C78A8', '#E67E22'],
    ax=ax
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)

ax.set_title('โครงการขนาดเล็กครองจำนวน แต่โครงการขนาดใหญ่มีผลต่อวงเงินรวม')
ax.set_xlabel('สัดส่วน (%)')
ax.set_ylabel('ช่วงวงเงินงบประมาณ (บาท)')
ax.legend(title='')
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_01_project_and_budget_share_by_budget_band.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


### สิ่งที่พบ: จำนวนโครงการกับวงเงินรวมอยู่คนละกลุ่ม

โครงการไม่เกิน 500,000 บาทคิดเป็น 76.34% ของจำนวนโครงการ แต่รวมกันประมาณ 8.23% ของวงเงินทั้งหมด ขณะที่โครงการมากกว่า 10 ล้านบาทมีเพียง 4.08% แต่ครองวงเงินประมาณ 67.31%

ผลนี้ทำให้เกิดคำถามว่าเหตุใดโครงการจำนวนมากจึงอยู่ไม่เกิน 500,000 บาท และขนาดดังกล่าวสัมพันธ์กับวิธีจัดซื้ออย่างไร


### 3.2 วิธีจัดซื้อสัมพันธ์กับขนาดโครงการอย่างไร

เมื่อพบการกระจุกของจำนวนโครงการไม่เกิน 500,000 บาท จึงเปรียบเทียบวิธีจัดซื้อก่อนนำกฎหมายมาอธิบาย

พระราชบัญญัติการจัดซื้อจัดจ้างฯ มาตรา 55 แบ่งวิธีหลัก และมาตรา 56 วรรคหนึ่ง (2)(ข) ประกอบกฎกระทรวงกำหนดวงเงินฯ อธิบายบริบทของวิธีเฉพาะเจาะจงสำหรับการจัดซื้อจัดจ้างทั่วไปวงเงินไม่เกิน 500,000 บาท

การอยู่ใต้เส้นดังกล่าวและใช้วิธีเฉพาะเจาะจงจึงเป็นรูปแบบที่กฎหมายรองรับ ไม่ใช่ความผิดปกติในตัวเอง


In [ ]:
method_column = (
    'ชื่อวิธีการจัดซื้อจัดจ้าง'
)

method_analysis = (
    project_data
    .assign(
        awarded_above_budget=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    budget_column
                ]
            )
        ),
        awarded_equal_budget=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    budget_column
                ]
            )
        ),
        reference_available=(
            project_data[
                reference_price_column
            ]
            .notna()
        ),
        awarded_above_reference=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    reference_price_column
                ]
            )
        ),
        awarded_equal_reference=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    reference_price_column
                ]
            )
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        ),
        above_budget_count=(
            'awarded_above_budget',
            'sum'
        ),
        equal_budget_count=(
            'awarded_equal_budget',
            'sum'
        ),
        reference_available_count=(
            'reference_available',
            'sum'
        ),
        above_reference_count=(
            'awarded_above_reference',
            'sum'
        ),
        equal_reference_count=(
            'awarded_equal_reference',
            'sum'
        )
    )
    .reset_index()
)

method_analysis['project_pct'] = (
    method_analysis['project_count']
    .div(
        method_analysis[
            'project_count'
        ].sum()
    )
    .mul(100)
)

method_analysis['budget_share_pct'] = (
    method_analysis['total_budget']
    .div(
        method_analysis[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

method_analysis['above_budget_pct'] = (
    method_analysis['above_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_budget_pct'] = (
    method_analysis['equal_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['above_reference_pct'] = (
    method_analysis['above_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_reference_pct'] = (
    method_analysis['equal_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis = (
    method_analysis
    .sort_values(
        'project_count',
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
display(
    method_analysis[
        [
            method_column,
            'project_count',
            'project_pct',
            'budget_share_pct',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct',
            'equal_budget_pct',
            'above_budget_pct',
            'equal_reference_pct',
            'above_reference_pct'
        ]
    ]
)

### สิ่งที่พบ: วิธีเฉพาะเจาะจงครองจำนวน ส่วน e-bidding ครองวงเงิน

วิธีเฉพาะเจาะจงมีประมาณ 76.35% ของจำนวนโครงการ แต่คิดเป็นประมาณ 9.33% ของวงเงินรวม ส่วน e-bidding มีประมาณ 21.22% ของจำนวน แต่ครองวงเงินประมาณ 83.34%

ผลนี้เชื่อมจุดกระจุกของโครงการขนาดเล็กกับบริบทวิธีจัดซื้อ และสนับสนุนให้เปรียบเทียบรูปแบบภายในวิธีเดียวกันก่อนข้ามวิธี


In [ ]:
method_label_map = {
    'เฉพาะเจาะจง': 'เฉพาะเจาะจง',
    'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-bidding',
    'คัดเลือก': 'คัดเลือก',
    'ตกลงราคา': 'ตกลงราคา'
}

method_plot = (
    method_analysis.loc[
        method_analysis['project_count'].gt(2)
    ].copy()
)

method_plot['procurement_method'] = (
    method_plot[method_column].map(method_label_map)
)

method_plot = (
    method_plot[
        ['procurement_method', 'project_pct', 'budget_share_pct']
    ]
    .melt(
        id_vars='procurement_method',
        var_name='measure',
        value_name='percentage'
    )
)

method_plot['measure'] = method_plot['measure'].map({
    'project_pct': 'สัดส่วนจำนวนโครงการ',
    'budget_share_pct': 'สัดส่วนวงเงินรวม'
})

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=method_plot,
    y='procurement_method',
    x='percentage',
    hue='measure',
    palette=['#4C78A8', '#E67E22'],
    ax=ax
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

ax.set_title('วิธีเฉพาะเจาะจงครองจำนวน ขณะที่ e-bidding ครองวงเงิน')
ax.set_xlabel('สัดส่วน (%)')
ax.set_ylabel('วิธีจัดซื้อจัดจ้าง')
ax.legend(title='')
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_02_project_and_budget_share_by_method.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


In [ ]:
budget_concentration = (
    project_data
    .assign(
        budget_le_500k=(
            project_data[
                budget_column
            ].le(500_000)
        ),
        budget_400k_to_500k=(
            project_data[
                budget_column
            ].gt(400_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_490k_to_500k=(
            project_data[
                budget_column
            ].ge(490_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_exactly_500k=(
            project_data[
                budget_column
            ].eq(500_000)
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        budget_le_500k_count=(
            'budget_le_500k',
            'sum'
        ),
        budget_400k_to_500k_count=(
            'budget_400k_to_500k',
            'sum'
        ),
        budget_490k_to_500k_count=(
            'budget_490k_to_500k',
            'sum'
        ),
        budget_exactly_500k_count=(
            'budget_exactly_500k',
            'sum'
        )
    )
    .reset_index()
)

count_columns = [
    'budget_le_500k_count',
    'budget_400k_to_500k_count',
    'budget_490k_to_500k_count',
    'budget_exactly_500k_count'
]

for column in count_columns:
    percentage_column = (
        column
        .replace('_count', '_pct')
    )

    budget_concentration[
        percentage_column
    ] = (
        budget_concentration[column]
        .div(
            budget_concentration[
                'project_count'
            ]
        )
        .mul(100)
    )

display(
    budget_concentration[
        [
            method_column,
            'project_count',
            'budget_le_500k_pct',
            'budget_400k_to_500k_pct',
            'budget_490k_to_500k_pct',
            'budget_exactly_500k_pct'
        ]
    ]
)

### กำหนดกลุ่มศึกษาหลัก

กฎหมายอธิบายเหตุผลเชิงโครงสร้างของเส้น 500,000 บาท แต่ไม่ได้ทำให้ทุกโครงการใต้เส้นเป็น anomaly กลุ่มศึกษาหลักจึงใช้เงื่อนไขร่วมกันสองข้อ:

1. วิธีเฉพาะเจาะจง
2. วงเงินไม่เกิน 500,000 บาท

ส่วนถัดไปแสดงว่าขอบเขตลดจากโครงการก่อสร้างทั้งหมดมาเหลือกลุ่มศึกษาอย่างไร


In [ ]:
study_population_mask = (
    project_data[method_column].eq('เฉพาะเจาะจง')
    & project_data[budget_column].le(500_000)
)

study_project_data = (
    project_data.loc[study_population_mask]
    .copy()
)

under_500k_count = (
    project_data[budget_column].le(500_000).sum()
)

study_population_summary = pd.DataFrame({
    'ขั้นการเลือกข้อมูล': [
        'โครงการจ้างก่อสร้างทั้งหมด',
        'วงเงินไม่เกิน 500,000 บาท',
        'วิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาท'
    ],
    'project_count': [
        len(project_data),
        under_500k_count,
        len(study_project_data)
    ]
})

study_population_summary['project_pct'] = (
    study_population_summary['project_count']
    / len(project_data)
    * 100
)

display(study_population_summary)


### จากประชากรอ้างอิงสู่กลุ่มศึกษาหลัก

การลดขอบเขตใช้หน่วยเดียวกันคือ “โครงการ” ตลอดทั้งภาพ ข้อมูลต้นทางระดับรายการถูกอธิบายใน Notebook 02 แล้ว จึงไม่นำจำนวนแถวมาปนกับจำนวนโครงการ


In [ ]:
scope_plot = (
    study_population_summary
    .sort_values('project_count', ascending=True)
    .copy()
)

scope_colors = [
    '#E67E22'
    if label == 'วิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาท'
    else '#4C78A8'
    for label in scope_plot['ขั้นการเลือกข้อมูล']
]

fig, ax = plt.subplots(figsize=(11, 5.5))

bars = ax.barh(
    scope_plot['ขั้นการเลือกข้อมูล'],
    scope_plot['project_count'],
    color=scope_colors
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} โครงการ ({percentage:.2f}%)'
        for count, percentage in zip(
            scope_plot['project_count'],
            scope_plot['project_pct']
        )
    ],
    padding=5
)

ax.set_title('การลดขอบเขตจากโครงการจ้างก่อสร้างทั้งหมดสู่กลุ่มศึกษาหลัก')
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')
ax.set_xlim(0, scope_plot['project_count'].max() * 1.22)
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_03_study_scope_selection.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


### 3.3 ข้อมูลกระจุกบริเวณใดรอบเส้น 500,000 บาท

แสดงช่วง 400,000–550,000 บาททั้งสองฝั่งของเส้นกฎหมาย โดยใช้ bin 10,000 บาท ทำเครื่องหมายเส้น 500,000 บาทและหน้าต่างสำรวจ 490,000–500,000 บาท

ภาพนี้ใช้ค้นหารูปแบบและตั้งคำถามต่อ ยังไม่ใช้การอยู่ใกล้เส้นเป็นตัวชี้วัดโดยลำพัง


In [ ]:
threshold_plot_data = (
    project_data.loc[
        project_data[budget_column].between(
            400_000,
            550_000,
            inclusive='both'
        ),
        [budget_column, method_column]
    ]
    .copy()
)

threshold_plot_data['method_group'] = np.where(
    threshold_plot_data[method_column].eq('เฉพาะเจาะจง'),
    'วิธีเฉพาะเจาะจง',
    'วิธีอื่น'
)

fig, ax = plt.subplots(figsize=(12, 6))

sns.histplot(
    data=threshold_plot_data,
    x=budget_column,
    hue='method_group',
    binwidth=10_000,
    multiple='stack',
    palette={
        'วิธีเฉพาะเจาะจง': '#4C78A8',
        'วิธีอื่น': '#B8BDC6'
    },
    ax=ax
)

ax.axvline(
    500_000,
    color='#D62728',
    linestyle='--',
    linewidth=2,
    label='เส้นแบ่งตามกฎหมาย 500,000 บาท'
)

ax.axvspan(
    490_000,
    500_000,
    color='#F2C14E',
    alpha=0.20,
    label='ช่วงสำรวจ 490,000–500,000 บาท'
)

ax.set_title('จำนวนโครงการกระจุกด้านล่างของเส้น 500,000 บาท')
ax.set_xlabel('วงเงินงบประมาณ (บาท)')
ax.set_ylabel('จำนวนโครงการ')
ax.ticklabel_format(style='plain', axis='x')
ax.spines[['top', 'right']].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title='')

plt.tight_layout()

figure_path = figure_directory / 'fig03_04_budget_distribution_around_500k.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'Figure saved: {figure_path}')


### สิ่งที่พบและการทดสอบถัดไป

วงเงินยอดนิยมหลายค่าอยู่บริเวณก่อนถึงเพดาน เช่น 490,000, 495,000, 498,000, 499,000 และ 500,000 บาท

500,000 บาทมาจากกฎหมาย ส่วน 490,000 บาทเป็นขอบล่างที่ทีมกำหนด จึงต้องเปรียบเทียบหลายหน้าต่างวงเงินและจำนวนขั้นต่ำของกลุ่มเกิดซ้ำใน Notebook 04 ก่อนเลือกเกณฑ์สุดท้าย


In [ ]:
near_threshold_windows = [
    (450_000, 500_000, '450,000–500,000'),
    (480_000, 500_000, '480,000–500,000'),
    (490_000, 500_000, '490,000–500,000'),
    (495_000, 500_000, '495,000–500,000')
]

near_threshold_sensitivity = pd.DataFrame([
    {
        'ช่วงวงเงิน': label,
        'project_count': (
            study_project_data[budget_column]
            .between(lower, upper, inclusive='both')
            .sum()
        )
    }
    for lower, upper, label in near_threshold_windows
])

near_threshold_sensitivity['project_pct'] = (
    near_threshold_sensitivity['project_count']
    / len(study_project_data)
    * 100
)

exact_500k_count = (
    study_project_data[budget_column].eq(500_000).sum()
)

display(near_threshold_sensitivity)
print(f'โครงการที่มีวงเงินเท่ากับ 500,000 บาท: {exact_500k_count:,}')


### 3.4 ราคาที่ตกลงสัมพันธ์กับงบประมาณและราคากลางอย่างไร

ใช้กลุ่มศึกษาหลักเป็นฐาน เปรียบเทียบราคาที่ตกลงกับงบประมาณและราคากลางแยกกัน เพื่อหลีกเลี่ยงกราฟที่ต้องอ่านอัตราส่วนสามมิติพร้อมกัน

เกณฑ์ส่วนต่างในส่วนนี้เป็นเกณฑ์สำรวจของการศึกษา ไม่ใช่เกณฑ์ตามกฎหมาย และจะทดสอบความไวเพิ่มเติมใน Notebook 04


In [ ]:
price_analysis_data = study_project_data.copy()

price_analysis_data['reference_to_budget_ratio'] = (
    price_analysis_data[reference_price_column]
    / price_analysis_data[budget_column]
)

price_analysis_data['usable_reference_price'] = (
    price_analysis_data[reference_price_column].notna()
    & price_analysis_data['reference_to_budget_ratio']
    .between(0.50, 1.50, inclusive='both')
)

budget_difference = (
    price_analysis_data[awarded_price_column]
    - price_analysis_data[budget_column]
)

reference_difference = (
    price_analysis_data[awarded_price_column]
    - price_analysis_data[reference_price_column]
)

price_analysis_data['budget_comparison'] = np.select(
    [
        budget_difference.lt(0),
        budget_difference.eq(0)
    ],
    ['ต่ำกว่า', 'เท่ากับ'],
    default='สูงกว่า'
)

price_analysis_data['reference_comparison'] = np.select(
    [
        reference_difference.lt(0),
        reference_difference.eq(0)
    ],
    ['ต่ำกว่า', 'เท่ากับ'],
    default='สูงกว่า'
)

comparison_order = ['ต่ำกว่า', 'เท่ากับ', 'สูงกว่า']

budget_comparison_summary = (
    price_analysis_data['budget_comparison']
    .value_counts()
    .reindex(comparison_order, fill_value=0)
)

reference_comparison_summary = (
    price_analysis_data.loc[
        price_analysis_data['usable_reference_price'],
        'reference_comparison'
    ]
    .value_counts()
    .reindex(comparison_order, fill_value=0)
)

price_comparison_summary = pd.DataFrame({
    'สถานะ': comparison_order,
    'เทียบวงเงินงบประมาณ': budget_comparison_summary.values,
    'เทียบราคากลางที่ใช้งานได้': reference_comparison_summary.values
})

display(price_comparison_summary)


In [ ]:
price_plot = (
    price_comparison_summary
    .set_index('สถานะ')
    .T
)

price_plot_pct = (
    price_plot
    .div(price_plot.sum(axis=1), axis=0)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(10, 4.8))

left = np.zeros(len(price_plot_pct))

colors = {
    'ต่ำกว่า': '#4C78A8',
    'เท่ากับ': '#B8BDC6',
    'สูงกว่า': '#D62728'
}

for status in comparison_order:
    values = price_plot_pct[status].values
    bars = ax.barh(
        price_plot_pct.index,
        values,
        left=left,
        label=status,
        color=colors[status]
    )

    labels = [
        f'{value:.1f}%' if value >= 3 else ''
        for value in values
    ]
    ax.bar_label(
        bars,
        labels=labels,
        label_type='center',
        color='white',
        fontsize=9
    )
    left += values

ax.set_title('ราคาที่ตกลงส่วนใหญ่อยู่ไม่สูงกว่าค่าเปรียบเทียบ')
ax.set_xlabel('สัดส่วนโครงการ (%)')
ax.set_ylabel('')
ax.set_xlim(0, 100)
ax.legend(title='ราคาที่ตกลง', ncol=3, loc='lower center', bbox_to_anchor=(0.5, -0.30))
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

plt.show()


### ข้อค้นพบจากการเปรียบเทียบราคา

ในกลุ่มศึกษาหลัก ราคาที่ตกลงต่ำกว่างบประมาณ 79,009 โครงการ เท่ากับงบประมาณ 56,804 โครงการ และสูงกว่า 257 โครงการ

สำหรับ 135,342 โครงการที่มีราคากลางใช้งานได้ ราคาที่ตกลงต่ำกว่าราคากลาง 96,558 โครงการ เท่ากับ 38,211 โครงการ และสูงกว่า 573 โครงการ

ส่วนใหญ่จึงไม่สูงกว่าค่าเปรียบเทียบ ขณะที่โครงการส่วนน้อยที่สูงกว่าจะส่งต่อไปทดสอบเกณฑ์จำนวนเงินและร้อยละใน Notebook 04


## 4. EDA เสริมสำหรับตรวจบริบท

ส่วนนี้เก็บการสำรวจหน่วยงาน พื้นที่ และผู้รับจ้างไว้ใน Notebook เพื่อใช้ตรวจบริบท แต่ไม่ใช้เป็นเส้นเรื่องหลักของ README เพราะไม่ได้ส่งต่อไปสร้างตัวชี้วัดโดยตรง


In [ ]:
agency_column = 'ชื่อหน่วยงาน'

agency_summary = (
    project_data
    .groupby(agency_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

agency_summary['project_share_pct'] = (
    agency_summary['project_count']
    .div(
        agency_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

agency_summary['budget_share_pct'] = (
    agency_summary['total_budget']
    .div(
        agency_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

agency_summary['total_budget_billion'] = (
    agency_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
top_agencies_by_count = (
    agency_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            agency_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

display(top_agencies_by_count)

In [ ]:
top_agencies_by_budget = (
    agency_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            agency_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

display(top_agencies_by_budget)

### ข้อค้นพบตามหน่วยงาน

กรมทางหลวงมีจำนวนโครงการมากที่สุด 5,774 โครงการ และมีวงเงินรวม
สูงที่สุดประมาณ 95.33 พันล้านบาท หรือ 20.03% ของวงเงินทั้งหมด
จึงเป็นหน่วยงานที่มีบทบาทสูงทั้งด้านจำนวนและมูลค่าโครงการ

กรมทางหลวงชนบทและกรมชลประทานอยู่ในอันดับสูงทั้งสองมิติเช่นกัน
ขณะที่กรมโยธาธิการและผังเมืองมีเพียง 780 โครงการ แต่มีวงเงินรวม
สูงถึง 32.31 พันล้านบาท และมีวงเงินมัธยฐาน 20 ล้านบาท
สะท้อนว่าเป็นหน่วยงานที่มีโครงการขนาดใหญ่โดยทั่วไป

ในทางตรงกันข้าม กรมการปกครองและกรมพัฒนาที่ดินมีจำนวนโครงการ
มากเป็นอันดับต้น แต่มีวงเงินมัธยฐานเพียง 373,000 และ 189,600 บาท
ตามลำดับ แสดงให้เห็นว่าการจัดอันดับจากจำนวนโครงการและวงเงินรวม
ให้ภาพที่แตกต่างกัน

หน่วยงาน 4 อันดับแรกตามวงเงินรวมครองวงเงินประมาณ 44.25%
ของข้อมูลทั้งหมด โดยส่วนใหญ่เป็นหน่วยงานที่รับผิดชอบ
โครงสร้างพื้นฐานด้านถนน น้ำ และงานโยธา

In [ ]:
agency_budget_plot = (
    agency_summary
    .nlargest(10, 'total_budget')
    .sort_values(
        'total_budget_billion',
        ascending=True
    )
    .copy()
)

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.barh(
    agency_budget_plot[agency_column],
    agency_budget_plot['total_budget_billion'],
    color='#4C78A8'
)

ax.bar_label(
    bars,
    labels=[
        f'{budget:.1f} พันล้าน'
        for budget in agency_budget_plot[
            'total_budget_billion'
        ]
    ],
    padding=3
)

ax.set_title('10 หน่วยงานที่มีวงเงินโครงการก่อสร้างรวมสูงสุด')
ax.set_xlabel('วงเงินรวม (พันล้านบาท)')
ax.set_ylabel('หน่วยงาน')
ax.spines[['top', 'right', 'left']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_05_top_agencies_by_construction_budget.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'บันทึกรูป: {figure_path}')


### 4.2 การกระจายตามพื้นที่

ส่วนนี้เปรียบเทียบจำนวนโครงการและวงเงินรวมในแต่ละจังหวัด
เพื่อดูว่าพื้นที่ที่มีโครงการจำนวนมากเป็นพื้นที่เดียวกับ
พื้นที่ที่ได้รับวงเงินรวมสูงหรือไม่

ข้อมูลจังหวัดระบุสถานที่ของโครงการตามชุดข้อมูลต้นทาง
จึงไม่ได้หมายถึงที่ตั้งสำนักงานใหญ่ของหน่วยงานหรือผู้รับจ้าง

In [ ]:
province_column = 'จังหวัด'

province_summary = (
    project_data
    .groupby(province_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

province_summary['project_share_pct'] = (
    province_summary['project_count']
    .div(
        province_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

province_summary['budget_share_pct'] = (
    province_summary['total_budget']
    .div(
        province_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

province_summary['total_budget_billion'] = (
    province_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            province_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            province_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

### ข้อค้นพบตามพื้นที่

นครราชสีมามีจำนวนโครงการมากที่สุด 8,641 โครงการ
รองลงมาคืออุบลราชธานี 7,181 โครงการ และขอนแก่น
6,605 โครงการ โดยจังหวัดที่มีโครงการจำนวนมากส่วนใหญ่
มีวงเงินมัธยฐานประมาณ 300,000–400,000 บาท

กรุงเทพมหานครมีจำนวน 4,825 โครงการ หรือ 2.70%
ของโครงการทั้งหมด แต่มีวงเงินรวมสูงถึง 177.54 พันล้านบาท
คิดเป็น 37.30% ของวงเงินทั้งหมด และมีวงเงินมัธยฐาน
4.46 ล้านบาท

ผลนี้แสดงว่าจังหวัดที่มีจำนวนโครงการมากที่สุดไม่จำเป็นต้องเป็น
จังหวัดที่มีวงเงินรวมสูงที่สุด โดยกรุงเทพมหานครมีโครงการ
ขนาดใหญ่กว่าจังหวัดอื่นโดยทั่วไป

การวิเคราะห์พื้นที่จะใช้เป็นบริบทประกอบการอธิบายข้อมูล
แต่จะไม่เป็นแกนหลักของตัวชี้วัดเพื่อการตรวจสอบ

### 4.3 การทำความเข้าใจข้อมูลผู้รับจ้าง

การวิเคราะห์ผู้รับจ้างใช้ข้อมูลทุกแถว เพราะหนึ่งโครงการอาจมีหลายสัญญา
หลายผู้ชนะ หรือเป็นกลุ่มผู้รับจ้างแบบ Joint Venture (JV)

ก่อนรวมมูลค่าตามผู้รับจ้าง ต้องตรวจความสัมพันธ์ระหว่างเลขประจำตัว
นิติบุคคลกับชื่อผู้ชนะ และแยกแถวสมาชิก JV เพื่อป้องกันการนับมูลค่าซ้ำ


In [ ]:
supplier_id_column = (
    'เลขประจำตัวนิติบุคคล 13 หลัก'
)

supplier_name_column = (
    'ชื่อผู้ชนะการเสนอราคา'
)

print('Most frequent supplier IDs:')

display(
    construction_data[
        supplier_id_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

print('\nMost frequent supplier names:')

display(
    construction_data[
        supplier_name_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

In [ ]:
supplier_id_profile = (
    construction_data
    .groupby(supplier_id_column)
    [supplier_name_column]
    .nunique()
)

supplier_name_profile = (
    construction_data
    .groupby(supplier_name_column)
    [supplier_id_column]
    .nunique()
)

supplier_relationship_summary = pd.Series({
    'Unique supplier IDs': (
        construction_data[
            supplier_id_column
        ].nunique()
    ),
    'Unique supplier names': (
        construction_data[
            supplier_name_column
        ].nunique()
    ),
    'Supplier IDs linked to multiple names': (
        supplier_id_profile.gt(1).sum()
    ),
    'Supplier names linked to multiple IDs': (
        supplier_name_profile.gt(1).sum()
    )
})

display(
    supplier_relationship_summary
    .to_frame(name='value')
)

### ข้อค้นพบเกี่ยวกับรหัสผู้รับจ้าง

ข้อมูลมีเลขประจำตัวนิติบุคคลที่ไม่ซ้ำ 33,685 ค่า
และชื่อผู้ชนะการเสนอราคาที่ไม่ซ้ำ 38,826 ค่า

พบเลขนิติบุคคล 4,234 รหัสที่เชื่อมโยงกับชื่อผู้ชนะมากกว่าหนึ่งรูปแบบ
ซึ่งอาจเกิดจากความแตกต่างของการสะกด คำนำหน้า สาขา
หรือรูปแบบการบันทึกชื่อ ขณะเดียวกันพบชื่อผู้ชนะ 253 ชื่อ
ที่เชื่อมโยงกับเลขนิติบุคคลมากกว่าหนึ่งรหัส

ดังนั้น การวิเคราะห์จะใช้เลขประจำตัวนิติบุคคลเป็นรหัสหลัก
และใช้ชื่อที่พบบ่อยที่สุดของแต่ละรหัสเป็นชื่อสำหรับแสดงผล
โดยไม่รวมผู้รับจ้างจากชื่อเพียงอย่างเดียว

In [ ]:
contract_budget_column = 'วงเงินงบประมาณในสัญญา (บาท)'

project_awarded_total = (
    project_data[
        awarded_price_column
    ].sum()
)

raw_contract_budget_total = (
    construction_data[
        contract_budget_column
    ].sum()
)

raw_value_difference = (
    raw_contract_budget_total
    - project_awarded_total
)

raw_value_reconciliation = pd.Series({
    'Project awarded total': (
        project_awarded_total
    ),
    'Raw contract budget total': (
        raw_contract_budget_total
    ),
    'Raw difference': (
        raw_value_difference
    ),
    'Raw difference pct': (
        raw_value_difference
        / project_awarded_total
        * 100
    ),
    'Missing contract budget rows': (
        construction_data[
            contract_budget_column
        ]
        .isna()
        .sum()
    )
})

display(
    raw_value_reconciliation
    .to_frame(name='value')
)

In [ ]:
raw_contract_value_by_project = (
    construction_data
    .groupby('รหัสโครงการ')[
        contract_budget_column
    ]
    .sum(min_count=1)
    .rename('raw_contract_budget_sum')
    .reset_index()
)

raw_project_value_check = (
    project_data[
        [
            'รหัสโครงการ',
            awarded_price_column
        ]
    ]
    .merge(
        raw_contract_value_by_project,
        on='รหัสโครงการ',
        how='left'
    )
)

raw_project_value_check['difference'] = (
    raw_project_value_check[
        'raw_contract_budget_sum'
    ]
    - raw_project_value_check[
        awarded_price_column
    ]
)

raw_project_value_check['is_matched'] = np.isclose(
    raw_project_value_check[
        'raw_contract_budget_sum'
    ],
    raw_project_value_check[
        awarded_price_column
    ],
    rtol=0,
    atol=1,
    equal_nan=False
)

raw_reconciliation_summary = pd.Series({
    'Projects checked': (
        len(raw_project_value_check)
    ),
    'Matched projects': (
        raw_project_value_check[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~raw_project_value_check[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        raw_project_value_check[
            'is_matched'
        ].mean()
        * 100
    )
})

display(
    raw_reconciliation_summary
    .to_frame(name='value')
)

### การตรวจพบโครงสร้าง Joint Venture (JV)

เมื่อรวมวงเงินทุกแถวโดยตรง ยอดระดับสัญญาตรงกับราคาที่ตกลง
178,616 จาก 178,978 โครงการ และพบส่วนต่าง 362 โครงการ
ยอดรวมดิบสูงกว่าระดับโครงการประมาณ 13.23 พันล้านบาท

บางโครงการมีทั้งแถวของกลุ่ม JV ซึ่งบันทึกมูลค่าสัญญาทั้งก้อน
และแถวบริษัทสมาชิกซึ่งบันทึกส่วนแบ่งของแต่ละราย หากรวมทุกแถว
พร้อมกันจะเกิดการนับมูลค่าซ้ำ

การวิเคราะห์ผู้รับจ้างจึงเก็บแถวของกลุ่ม JV และไม่นับแถวสมาชิกซ้ำ
ก่อนตรวจสอบยอดอีกครั้ง


In [ ]:
construction_data[
    'is_joint_venture_member'
] = (
    construction_data[
        supplier_name_column
    ]
    .astype('string')
    .str.contains(
        'สัญญากิจการค้าร่วม',
        na=False
    )
)

joint_venture_member_summary = pd.Series({
    'Joint-venture member rows': (
        construction_data[
            'is_joint_venture_member'
        ].sum()
    ),
    'Projects with member rows': (
        construction_data.loc[
            construction_data[
                'is_joint_venture_member'
            ],
            'รหัสโครงการ'
        ].nunique()
    ),
    'Member-row contract value': (
        construction_data.loc[
            construction_data[
                'is_joint_venture_member'
            ],
            contract_budget_column
        ].sum()
    )
})

display(
    joint_venture_member_summary
    .to_frame(name='value')
)

In [ ]:
supplier_entity_data = (
    construction_data
    .loc[
        ~construction_data[
            'is_joint_venture_member'
        ]
    ]
    .copy()
)

entity_value_by_project = (
    supplier_entity_data
    .groupby('รหัสโครงการ')[
        contract_budget_column
    ]
    .sum(min_count=1)
    .rename('entity_contract_value_sum')
    .reset_index()
)

entity_reconciliation = (
    project_data[
        [
            'รหัสโครงการ',
            awarded_price_column
        ]
    ]
    .merge(
        entity_value_by_project,
        on='รหัสโครงการ',
        how='left'
    )
)

entity_reconciliation['difference'] = (
    entity_reconciliation[
        'entity_contract_value_sum'
    ]
    - entity_reconciliation[
        awarded_price_column
    ]
)

entity_reconciliation['is_matched'] = np.isclose(
    entity_reconciliation[
        'entity_contract_value_sum'
    ],
    entity_reconciliation[
        awarded_price_column
    ],
    rtol=0,
    atol=1,
    equal_nan=False
)

entity_reconciliation_summary = pd.Series({
    'Supplier entity rows': (
        len(supplier_entity_data)
    ),
    'Projects checked': (
        len(entity_reconciliation)
    ),
    'Matched projects': (
        entity_reconciliation[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~entity_reconciliation[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        entity_reconciliation[
            'is_matched'
        ].mean()
        * 100
    ),
    'Total absolute difference': (
        entity_reconciliation[
            'difference'
        ].abs().sum()
    )
})

display(
    entity_reconciliation_summary
    .to_frame(name='value')
)

In [ ]:
matched_entity_project_ids = set(
    entity_reconciliation.loc[
        entity_reconciliation[
            'is_matched'
        ],
        'รหัสโครงการ'
    ]
)

supplier_analysis_data = (
    supplier_entity_data
    .loc[
        supplier_entity_data[
            'รหัสโครงการ'
        ].isin(
            matched_entity_project_ids
        )
        & supplier_entity_data[
            contract_budget_column
        ].notna()
    ]
    .copy()
)

print(
    f'Supplier entity rows used: '
    f'{len(supplier_analysis_data):,}'
)

print(
    f'Reconciled projects used: '
    f'{supplier_analysis_data["รหัสโครงการ"].nunique():,}'
)

In [ ]:
supplier_name_lookup = (
    supplier_analysis_data
    .groupby(
        [
            supplier_id_column,
            supplier_name_column
        ]
    )
    .size()
    .rename('name_count')
    .reset_index()
    .sort_values(
        [
            supplier_id_column,
            'name_count',
            supplier_name_column
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .drop_duplicates(
        subset=supplier_id_column,
        keep='first'
    )[
        [
            supplier_id_column,
            supplier_name_column
        ]
    ]
    .rename(
        columns={
            supplier_name_column:
            'supplier_display_name'
        }
    )
)

supplier_summary = (
    supplier_analysis_data
    .groupby(supplier_id_column)
    .agg(
        supplier_project_count=(
            'รหัสโครงการ',
            'nunique'
        ),
        supplier_record_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_contract_value=(
            contract_budget_column,
            'sum'
        ),
        median_contract_value=(
            contract_budget_column,
            'median'
        )
    )
    .reset_index()
    .merge(
        supplier_name_lookup,
        on=supplier_id_column,
        how='left'
    )
)

supplier_summary[
    'total_contract_value_million'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(1_000_000)
)

supplier_summary[
    'value_share_pct'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(
        supplier_summary[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

In [ ]:
top_suppliers_by_projects = (
    supplier_summary
    .nlargest(
        15,
        'supplier_project_count'
    )[
        [
            supplier_id_column,
            'supplier_display_name',
            'supplier_project_count',
            'supplier_record_count',
            'total_contract_value_million'
        ]
    ]
)

top_suppliers_by_value = (
    supplier_summary
    .nlargest(
        15,
        'total_contract_value'
    )[
        [
            supplier_id_column,
            'supplier_display_name',
            'total_contract_value_million',
            'value_share_pct',
            'supplier_project_count',
            'median_contract_value'
        ]
    ]
)

display(top_suppliers_by_projects)
display(top_suppliers_by_value)

### ข้อค้นพบเกี่ยวกับผู้รับจ้าง

พบแถวสมาชิก JV 710 แถว ครอบคลุม 362 โครงการ และมีมูลค่ารวม
ประมาณ 13.06 พันล้านบาท หลังไม่นับแถวสมาชิกซ้ำ ยอดตรงกัน
178,967 จาก 178,978 โครงการ หรือ 99.994% เหลือ 11 โครงการ
ที่ยอดยังไม่ตรงกัน

ผู้รับจ้างที่ได้โครงการจำนวนมากที่สุดไม่ใช่รายที่ได้มูลค่ารวมสูงที่สุด
จึงต้องพิจารณาทั้งจำนวนโครงการและมูลค่าสัญญาควบคู่กัน


In [ ]:
supplier_pareto = (
    supplier_summary
    .sort_values(
        'total_contract_value',
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

supplier_pareto['supplier_rank'] = (
    supplier_pareto.index + 1
)

supplier_pareto['cumulative_value_pct'] = (
    supplier_pareto[
        'total_contract_value'
    ]
    .cumsum()
    .div(
        supplier_pareto[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

total_suppliers = len(
    supplier_pareto
)

supplier_count_50 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(50)
    .idxmax()
    + 1
)

supplier_count_80 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(80)
    .idxmax()
    + 1
)

supplier_count_90 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(90)
    .idxmax()
    + 1
)

In [ ]:
supplier_concentration_summary = pd.Series({
    'Supplier entities': (
        total_suppliers
    ),
    'Top 1 value share (%)': (
        supplier_pareto
        .head(1)[
            'value_share_pct'
        ].sum()
    ),
    'Top 10 value share (%)': (
        supplier_pareto
        .head(10)[
            'value_share_pct'
        ].sum()
    ),
    'Top 100 value share (%)': (
        supplier_pareto
        .head(100)[
            'value_share_pct'
        ].sum()
    ),
    'Suppliers accounting for 50%': (
        supplier_count_50
    ),
    'Supplier pct accounting for 50%': (
        supplier_count_50
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 80%': (
        supplier_count_80
    ),
    'Supplier pct accounting for 80%': (
        supplier_count_80
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 90%': (
        supplier_count_90
    ),
    'Supplier pct accounting for 90%': (
        supplier_count_90
        / total_suppliers
        * 100
    )
})

display(
    supplier_concentration_summary
    .to_frame(name='value')
)

### ข้อค้นพบเกี่ยวกับการกระจายมูลค่าตามผู้รับจ้าง

หลังไม่นับแถวสมาชิก JV ซ้ำ พบผู้รับจ้างหรือกลุ่มผู้รับจ้าง 33,570 ราย รายใหญ่ที่สุดครองมูลค่า 1.51% และ 100 อันดับแรกครอง 27.83%

ผู้รับจ้าง 1,870 ราย หรือ 5.57% ครองมูลค่าสะสม 80% แต่ไม่มีรายเดียวครองภาพรวมประเทศสูงมาก ผลระดับประเทศจึงใช้เป็นบริบท ก่อนเปลี่ยนไปวิเคราะห์ความสัมพันธ์ภายในหน่วยงานย่อยใน Notebook 04


In [ ]:
supplier_pareto[
    'cumulative_supplier_pct'
] = (
    supplier_pareto[
        'supplier_rank'
    ]
    .div(total_suppliers)
    .mul(100)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    supplier_pareto['cumulative_supplier_pct'],
    supplier_pareto['cumulative_value_pct'],
    color='#4C78A8',
    linewidth=2
)

supplier_pct_80 = (
    supplier_count_80
    / total_suppliers
    * 100
)

ax.axhline(
    y=80,
    color='#D62728',
    linestyle='--',
    linewidth=1.5,
    label='มูลค่าสัญญาสะสม 80%'
)

ax.axvline(
    x=supplier_pct_80,
    color='#E67E22',
    linestyle='--',
    linewidth=1.5,
    label=(
        f'ผู้รับจ้าง {supplier_count_80:,} ราย '
        f'({supplier_pct_80:.1f}%)'
    )
)

ax.scatter(
    supplier_pct_80,
    80,
    color='#D62728',
    s=60,
    zorder=3
)

ax.set_title('มูลค่าสัญญาสะสมกระจุกอยู่ในผู้รับจ้างส่วนน้อย')
ax.set_xlabel('สัดส่วนผู้รับจ้างสะสม (%)')
ax.set_ylabel('สัดส่วนมูลค่าสัญญาสะสม (%)')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_06_cumulative_contract_value_by_supplier.png'
fig.savefig(figure_path, dpi=150, bbox_inches='tight')

plt.show()
print(f'บันทึกรูป: {figure_path}')


## 5. สรุป EDA

- โครงการไม่เกิน 500,000 บาทครอง 76.34% ของจำนวน แต่เพียง 8.23% ของวงเงินรวม
- วิธีเฉพาะเจาะจงครองจำนวน ขณะที่ e-bidding ครองวงเงิน
- กลุ่มศึกษาหลักมี 136,070 โครงการ
- ช่วง 490,000–500,000 บาทมี 26,154 โครงการ แต่เป็นช่วงวิเคราะห์ของทีม ไม่ใช่เกณฑ์ตามกฎหมาย
- ราคาที่ตกลงส่วนใหญ่ไม่สูงกว่างบประมาณหรือราคากลาง
- Notebook 04 จะทดสอบความไวของ threshold และตรวจการซ้อนทับของรูปแบบเกิดซ้ำ การพึ่งพาผู้รับจ้าง และส่วนต่างราคา


In [ ]:
project_output_path = (
    data_path.parent
    / 'construction_projects_2569.csv'
)

project_data.to_csv(
    project_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(
    f'Project-level data saved to:\n'
    f'{project_output_path}'
)

In [ ]:
supplier_output_path = (
    data_path.parent
    / 'construction_supplier_summary_2569.csv'
)

supplier_summary.to_csv(
    supplier_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(
    f'Supplier summary saved to:\n'
    f'{supplier_output_path}'
)

In [ ]:
output_files = [
    project_output_path,
    supplier_output_path
]

for file_path in output_files:
    print(
        f'{file_path.name}: '
        f'{file_path.stat().st_size / (1024 ** 2):,.2f} MB'
    )

## 6. ไฟล์ผลลัพธ์

Notebook สร้างไฟล์:

1. `construction_projects_2569.csv` — ข้อมูลหนึ่งแถวต่อโครงการก่อสร้างทั้งหมด พร้อมตัวแปรสำหรับระบุกลุ่มศึกษาหลัก
2. `construction_supplier_summary_2569.csv` — สรุปผู้รับจ้างหรือกลุ่มผู้รับจ้างหลังไม่นับแถวสมาชิก JV ซ้ำ

Notebook 04 จะกำหนดกลุ่มศึกษาหลักจากวิธีเฉพาะเจาะจงและวงเงินไม่เกิน 500,000 บาทอีกครั้ง เพื่อให้เงื่อนไขตรวจสอบย้อนกลับได้
